# Data Preprocessing Notebook

**Project**: Intelligent Employee Task Allocation — DS-20 (Industry Explorer Track)  
**Client**: Cygnus One (Pvt) Ltd  
**Input**: `data/raw/Combined_Employee_Task_Data.csv` — 11,633 rows × 22 columns  
**Output**: `data/processed/preprocessed_employee_task_data.csv`  

---

## 📌 Purpose of This Notebook

Transform the combined raw dataset into a clean, consistently formatted dataset that is ready for **Feature Engineering** and **Model Training**.

Decisions made in this notebook are directly informed by findings from `02_Data_EDA.ipynb`, including:
- Missing-value patterns identified in EDA Step 2
- Data quality issues uncovered in EDA Step 2
- Leakage risks flagged in EDA Step 9 and Step 11
- Redundant column relationships confirmed in EDA Step 1 and Step 7

---

## 📋 Preprocessing Workflow Overview

| Step | Task |
|---|---|
| 1 | Load the combined dataset & create a working copy |
| 2 | Remove unnecessary / redundant columns |
| 3 | Handle missing values |
| 4 | Fix data types |
| 5 | Handle duplicate records |
| 6 | Clean categorical variables |
| 7 | Clean numerical variables |
| 8 | Clean date columns |
| 9 | Clean text columns |
| 10 | Validate employee and task identifiers |
| 11 | Handle data leakage risks |
| 12 | Final data quality validation |
| 13 | Save the preprocessed dataset |

---

## 📦 Import Libraries

In [1]:
import pandas as pd

---

## Step 1 — Load the Combined Dataset

Load the 11,633 × 22 combined dataset produced by `01_Data_Preparation.ipynb`.

**Actions:**
- Load `Combined_Employee_Task_Data.csv` from `data/raw/`
- Create a **working copy** (`df`) so `df_raw` is always available for comparison or rollback
- Confirm shape and column names

> ⚠️ `df_raw` is kept **unchanged** throughout this notebook. All modifications are applied only to `df`.

In [2]:
file_path = '../data/raw/Combined_Employee_Task_Data.csv'
df_raw = pd.read_csv(file_path)
df = df_raw.copy()
print(f"Working Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("Column Names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col}")

Working Dataset Shape: 11633 rows, 22 columns

Column Names:
01. Timesheet_ID
02. Date
03. Task_ID
04. Task_Name
05. Work_Description
06. Hours_Spent
07. Project_Name
08. Employee_ID
09. Employee_Name
10. Employee_Department
11. Employee_Job_Position
12. Task_Description
13. Timesheet_Work_Logs
14. Original_Task_Description
15. Task_Priority
16. Estimated_Planned_Hours
17. Actual_Hours_Spent
18. Timesheet_Logs_Count
19. Task_Stage
20. Created_Date
21. Deadline_Date
22. All_Collaborating_Employees


---

## Step 2 — Remove Unnecessary / Redundant Columns

Based on findings from the EDA, columns that are redundant, non-predictive, or high-risk are identified and either dropped or flagged.

### ❌ Columns Dropped

| Column | Reason |
|---|---|
| `Timesheet_ID` | Unique row identifier — carries no predictive value for the model |
| `Employee_Name` | Perfectly correlated with `Employee_ID` (the target variable) — using it would cause **data leakage** |
| `Original_Task_Description` | High null rate (>99% missing, confirmed in EDA Step 2); overshadowed by the richer `Task_Description` field |

### 🚩 Columns Flagged (Retained for now, removed before modelling)

These columns contain **post-assignment information** — data that would not exist at the moment a manager wants to assign a new task. They are not dropped yet, but are clearly documented for removal before the train/test split.

| Flagged Column | Why It's a Leakage Risk |
|---|---|
| `Actual_Hours_Spent` | Only known **after** the task has been worked on — unknown at assignment time |
| `Task_Stage` | Reflects the current/final state (e.g., `Done`) — implies post-assignment status |
| `Timesheet_Logs_Count` | Aggregated after timesheets are submitted — post-assignment metric |
| `Timesheet_Work_Logs` | Descriptions written **after** the work is performed |
| `All_Collaborating_Employees` | Unknown until all employees have logged time on the task |

> 📌 **EDA Reference**: Leakage risks were first identified in EDA Step 9 (Target Variable Analysis) and EDA Step 11 (Data Integration Validation).

In [3]:
columns_to_drop = [
    'Timesheet_ID',
    'Employee_Name',
    'Original_Task_Description'
]
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')
leakage_risk_columns = [
    'Actual_Hours_Spent',
    'Task_Stage',
    'Timesheet_Logs_Count',
    'Timesheet_Work_Logs',
    'All_Collaborating_Employees'
]
print("--- Step 2: Column Removal & Leakage Flagging ---")
print(f"Columns dropped: {columns_to_drop}")
print(f"New dataset shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("⚠️ FLAGGED FOR POTENTIAL LEAKAGE (Retained for now):")
for col in leakage_risk_columns:
    print(f" - {col}")

--- Step 2: Column Removal & Leakage Flagging ---
Columns dropped: ['Timesheet_ID', 'Employee_Name', 'Original_Task_Description']
New dataset shape: 11633 rows, 19 columns

⚠️ FLAGGED FOR POTENTIAL LEAKAGE (Retained for now):
 - Actual_Hours_Spent
 - Task_Stage
 - Timesheet_Logs_Count
 - Timesheet_Work_Logs
 - All_Collaborating_Employees


---

## Step 3 — Handle Missing Values

From EDA Step 2 (Data Quality Analysis), we know the following columns have missing values:

| Column | Missing % (from EDA) | Strategy | Rationale |
|---|---|---|---|
| `Work_Description` | ~0.01% | Replace with empty string `""` | NLP pipelines (TF-IDF) require strings, not nulls |
| `Task_Description` | ~0% | Replace with empty string `""` | Same as above |
| `Timesheet_Work_Logs` | ~0.6% | Replace with empty string `""` | Same as above |
| `All_Collaborating_Employees` | ~0% | Replace with empty string `""` | Same as above |
| `Employee_Department` | ~0.7% | Replace with `"Unknown"` | Preserves the row; model can learn from this pattern |
| `Project_Name` | ~0% | Replace with `"Unknown"` | Categorical — keep the row |
| `Task_Priority` | ~0% | Replace with `"Unknown"` | Categorical — keep the row |
| `Task_Stage` | ~0% | Replace with `"Unknown"` | Categorical — keep the row |
| `Hours_Spent` | ~0% | Fill with `0.0` | Missing hours logically implies 0 hours logged |
| `Estimated_Planned_Hours` | ~0% | Fill with `0.0` | Missing estimate implies no estimate was set |
| `Actual_Hours_Spent` | ~0% | Fill with `0.0` | Missing actual hours implies 0 |
| `Timesheet_Logs_Count` | ~0% | Fill with `0.0` | Missing count implies 0 logs |
| `Employee_ID` | None expected | Drop row if missing | Target variable — row is unusable without it |
| `Task_ID` | None expected | Drop row if missing | Join key — row is unusable without it |
| `Deadline_Date` | ~86% missing | Leave as `NaT` | A missing deadline is a valid business state — not an error |
| `Date` / `Created_Date` | None expected | Drop row if missing | Essential timestamps — row is unusable without them |

> 📌 **EDA Reference**: Missing value rates were confirmed in EDA Step 2 (Data Quality Analysis), specifically the missing-value bar chart and null-count table.

In [4]:
print("--- Missing Values Before Treatment ---")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\n")

--- Missing Values Before Treatment ---
Work_Description           1
Employee_Department       77
Timesheet_Work_Logs       74
Deadline_Date          10012
dtype: int64




In [5]:
text_columns = ['Work_Description', 'Task_Description', 'Timesheet_Work_Logs', 'All_Collaborating_Employees']
for col in text_columns:
    if col in df.columns:
        df[col] = df[col].fillna("")

In [6]:
categorical_columns = ['Project_Name', 'Task_Name', 'Task_Priority', 'Task_Stage', 'Employee_Department', 'Employee_Job_Position']
for col in categorical_columns:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

In [7]:
numerical_columns = ['Hours_Spent', 'Estimated_Planned_Hours', 'Actual_Hours_Spent', 'Timesheet_Logs_Count']
for col in numerical_columns:
    if col in df.columns:
        df[col] = df[col].fillna(0.0)

In [8]:
if df['Employee_ID'].isnull().sum() > 0:
    print(f"Dropping {df['Employee_ID'].isnull().sum()} rows due to missing Employee_ID (Target).")
    df.dropna(subset=['Employee_ID'], inplace=True)
if df['Task_ID'].isnull().sum() > 0:
    print(f"Dropping {df['Task_ID'].isnull().sum()} rows due to missing Task_ID.")
    df.dropna(subset=['Task_ID'], inplace=True)

In [9]:
print("--- Missing Values After Treatment (Excluding Dates) ---")
missing_after = df.drop(columns=['Created_Date', 'Deadline_Date'], errors='ignore').isnull().sum()
print(missing_after[missing_after > 0].to_string() if missing_after.sum() > 0 else "No missing values remaining in processed columns!")

--- Missing Values After Treatment (Excluding Dates) ---
No missing values remaining in processed columns!


---

## Step 4 — Fix Data Types

Convert each column into the correct Python/Pandas data type to ensure consistency, correct calculations, and compatibility with ML libraries.

### Type Conversion Plan

| Column(s) | Target Type | Notes |
|---|---|---|
| `Date`, `Created_Date`, `Deadline_Date` | `datetime64` | `errors='coerce'` converts invalid values to `NaT` safely |
| `Hours_Spent`, `Estimated_Planned_Hours`, `Actual_Hours_Spent` | `float64` | Numeric measurements |
| `Timesheet_Logs_Count` | `float64` / `int64` | Count variable — numeric |
| `Task_ID`, `Employee_ID` | `str` (object) | Identifiers — must **not** be treated as numbers |

> ⚠️ IDs are **kept as strings**. Converting them to integers could imply a numerical relationship that does not exist (e.g., `EMP-11 > EMP-10`).

In [10]:
date_columns = ['Date', 'Created_Date', 'Deadline_Date']
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
numeric_columns = [
    'Hours_Spent', 
    'Estimated_Planned_Hours', 
    'Actual_Hours_Spent', 
    'Timesheet_Logs_Count'
]
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
id_columns = ['Task_ID', 'Employee_ID']
for col in id_columns:
    if col in df.columns:
        df[col] = df[col].astype(str)
print("--- Step 4: Data Types After Conversion ---")
print(df[date_columns + numeric_columns + id_columns].dtypes)

--- Step 4: Data Types After Conversion ---
Date                       datetime64[us]
Created_Date               datetime64[us]
Deadline_Date              datetime64[us]
Hours_Spent                       float64
Estimated_Planned_Hours           float64
Actual_Hours_Spent                float64
Timesheet_Logs_Count                int64
Task_ID                               str
Employee_ID                           str
dtype: object


---

## Step 5 — Handle Duplicate Records

Check and remove duplicate records. The key distinction here is **what kind of duplicate** we are dealing with.

### Duplicate Strategy

| Scenario | Action | Reason |
|---|---|---|
| Completely duplicated rows (all 22 columns identical) | Drop — keep first | Accidental double-entry of the same timesheet log |
| Duplicate `Timesheet_ID` | Drop — keep first | Each timesheet entry must be unique |
| Same `Task_ID` appearing multiple times | **Do NOT drop** | One task legitimately has multiple timesheet log entries |

> ⚠️ **Important**: We deliberately do **not** use `drop_duplicates('Task_ID')`. The same task can appear across many rows because multiple employees can log hours against it on different dates.

> 📌 **EDA Reference**: EDA Step 2 confirmed the existence of tasks with multiple timesheet records and multiple associated employees.

In [11]:
print("--- Step 5: Duplicate Handling ---")
initial_rows = df.shape[0]
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} completely duplicated rows. Dropping them...")
    df.drop_duplicates(inplace=True)
else:
    print("No completely duplicated rows found.")
if 'Timesheet_ID' in df.columns:
    duplicate_ts_ids = df.duplicated(subset=['Timesheet_ID']).sum()
    if duplicate_ts_ids > 0:
        print(f"Found {duplicate_ts_ids} duplicate Timesheet_IDs. Keeping the first occurrence...")
        df.drop_duplicates(subset=['Timesheet_ID'], keep='first', inplace=True)
    else:
        print("No duplicate Timesheet_IDs found.")
else:
    print("Timesheet_ID was removed in Step 2; skipping Timesheet_ID duplicate check.")
rows_dropped = initial_rows - df.shape[0]
print(f"\nTotal rows dropped in Step 5: {rows_dropped}")
print(f"Dataset shape after Step 5: {df.shape[0]} rows, {df.shape[1]} columns")

--- Step 5: Duplicate Handling ---
Found 3 completely duplicated rows. Dropping them...
Timesheet_ID was removed in Step 2; skipping Timesheet_ID duplicate check.

Total rows dropped in Step 5: 3
Dataset shape after Step 5: 11630 rows, 19 columns


---

## Step 6 — Clean Categorical Variables

Standardize the categorical columns to eliminate inconsistencies caused by formatting differences.

**Target columns:**
- `Employee_Department`
- `Employee_Job_Position`
- `Task_Priority`
- `Task_Stage`
- `Project_Name`

### Cleaning Steps Applied

| Issue | Fix Applied |
|---|---|
| Leading/trailing spaces (e.g., `" Developer "`) | `.str.strip()` |
| Inconsistent capitalisation (e.g., `"developer"`, `"DEVELOPER"`) | `.str.title()` — Title Case |
| Empty strings after stripping | Replaced with `"Unknown"` |
| `'nan'` / `'None'` strings (from previous fillna) | Replaced with `"Unknown"` |
| Known abbreviations or alternate spellings (e.g., `'Dev'` → `'Developer'`) | Explicit mapping dictionary |

> 📌 **EDA Reference**: EDA Step 3 (Employee Analysis) and EDA Step 4 (Task Analysis) revealed inconsistent category names through `.unique()` inspections of department and job-position columns.

In [12]:
print("--- Step 6: Cleaning Categorical Variables ---")
categorical_cols = [
    'Employee_Department', 
    'Employee_Job_Position', 
    'Task_Priority', 
    'Task_Stage', 
    'Project_Name'
]
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()
        df[col] = df[col].replace({'': 'Unknown', 'Nan': 'Unknown', 'None': 'Unknown'})
job_mapping = {
    'Dev': 'Developer',
    'Qa': 'Quality Assurance',
    'Project Mgr': 'Project Manager'
}
department_mapping = {
    'R&D': 'Research And Development (R&D)',
    'Research & Development': 'Research And Development (R&D)'
}
if 'Employee_Job_Position' in df.columns:
    df['Employee_Job_Position'] = df['Employee_Job_Position'].replace(job_mapping)
if 'Employee_Department' in df.columns:
    df['Employee_Department'] = df['Employee_Department'].replace(department_mapping)
for col in categorical_cols:
    if col in df.columns:
        print(f"\nUnique categories in '{col}' ({df[col].nunique()} total):")
        print(df[col].unique()[:10])

--- Step 6: Cleaning Categorical Variables ---

Unique categories in 'Employee_Department' (8 total):
<ArrowStringArray>
['Research And Development (R&D)',                 'Colombo Branch',
              'Business Solution',          'Support & Maintenance',
              'Sales & Marketing',                        'Unknown',
         'Quality Assurance (Qa)',                 'Administration']
Length: 8, dtype: str

Unique categories in 'Employee_Job_Position' (20 total):
<ArrowStringArray>
['Team Lead - Research And Development (R&D)',
                            'Project Manager',
            'Associate Functional Consultant',
                'Associate Software Engineer',
    'Training Software Engineer - Internship',
                             'Support Intern',
        'Training Odoo Functional Consultant',
                          'Software Engineer',
               'Functional Support Executive',
                 'Odoo Functional Consultant']
Length: 10, dtype: str

Unique cat

---

## Step 7 — Clean Numerical Variables

Inspect and correct issues in the numerical columns, without automatically deleting outliers.

**Target columns:**
- `Hours_Spent`
- `Estimated_Planned_Hours`
- `Actual_Hours_Spent`
- `Timesheet_Logs_Count`

### Issues Checked & Actions Taken

| Issue | Action | Rationale |
|---|---|---|
| Negative values | Convert to absolute value (`.abs()`) | Time and counts cannot be negative — likely a data-entry sign error |
| `Hours_Spent > 24` | Cap at `24.0` | A single timesheet entry cannot exceed 24 hours in one day — physically impossible |
| Zero values in `Hours_Spent` | **Kept as-is** | Zero hours is valid — confirmed in EDA (e.g., meetings logged at 0 hours) |
| Large values in `Actual_Hours_Spent` (e.g., 1,070 hours) | **Kept as-is** | EDA confirmed these are genuine long-running tasks, not errors |
| Remaining missing numerics | Fill with `0.0` | Catch-all safety net after Steps 3 and 4 |

> ⚠️ **Outliers are not automatically deleted.** EDA Step 5 (Time & Effort Analysis) confirmed that high values in `Actual_Hours_Spent` and `Timesheet_Logs_Count` are genuine observations from long-running projects, not data errors.

In [13]:
print("--- Step 7: Cleaning Numerical Variables ---")
numerical_cols = [
    'Hours_Spent', 
    'Estimated_Planned_Hours', 
    'Actual_Hours_Spent', 
    'Timesheet_Logs_Count'
]
for col in numerical_cols:
    if col in df.columns:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            print(f"Fixing {neg_count} negative values in '{col}' (converting to absolute value).")
            df[col] = df[col].abs()
if 'Hours_Spent' in df.columns:
    unrealistic_hours = (df['Hours_Spent'] > 24).sum()
    if unrealistic_hours > 0:
        print(f"Capping {unrealistic_hours} unrealistic 'Hours_Spent' values at 24.0 hours.")
        df['Hours_Spent'] = df['Hours_Spent'].clip(upper=24.0)
missing_num = df[numerical_cols].isnull().sum().sum()
if missing_num > 0:
    print(f"Warning: Found {missing_num} missing numerical values. Filling with 0.0.")
    df[numerical_cols] = df[numerical_cols].fillna(0.0)
print("\nNumerical Data Summary (Min, Max, Mean) after Cleaning:")
print(df[numerical_cols].describe().T[['min', 'max', 'mean']])

--- Step 7: Cleaning Numerical Variables ---

Numerical Data Summary (Min, Max, Mean) after Cleaning:
                         min      max        mean
Hours_Spent              0.0    24.00    3.613436
Estimated_Planned_Hours  0.0   552.00   10.156364
Actual_Hours_Spent       0.0  1070.87  131.141189
Timesheet_Logs_Count     1.0   441.00   50.163801


---

## Step 8 — Clean Date Columns

Standardize the three date columns and fix any invalid date values.

**Target columns:**
- `Date` — when the timesheet entry was logged
- `Created_Date` — when the task was created
- `Deadline_Date` — task deadline (optional — many nulls expected)

### Cleaning Actions

| Action | Column(s) | Notes |
|---|---|---|
| Re-confirm datetime conversion | All three | Safety check following Step 4 — `errors='coerce'` → `NaT` |
| Drop rows with missing essential dates | `Date`, `Created_Date` | These timestamps are mandatory for any timesheet entry |
| Leave missing `Deadline_Date` as `NaT` | `Deadline_Date` | A missing deadline is a valid business state — not an error |
| Fix chronological errors (deadline < creation) | `Deadline_Date` | Where `Deadline_Date < Created_Date`, set deadline to `NaT` (treat as unknown) |

> 📌 **No date-derived features are created here.** Variables such as:
> - `task_age` (days since creation)
> - `days_to_deadline`
> - `month`, `weekday`
>
> belong in the **Feature Engineering notebook**, not preprocessing.

In [14]:
print("--- Step 8: Cleaning Date Columns ---")
date_cols = ['Date', 'Created_Date', 'Deadline_Date']
for col in date_cols:
    if col in df.columns and not pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col] = pd.to_datetime(df[col], errors='coerce')
for col in ['Date', 'Created_Date']:
    if col in df.columns:
        missing_dates = df[col].isnull().sum()
        if missing_dates > 0:
            print(f"Dropping {missing_dates} rows due to missing essential '{col}'.")
            df.dropna(subset=[col], inplace=True)
if 'Created_Date' in df.columns and 'Deadline_Date' in df.columns:
    invalid_deadlines = df['Deadline_Date'] < df['Created_Date']
    invalid_count = invalid_deadlines.sum()
    if invalid_count > 0:
        print(f"Fixing {invalid_count} records where Deadline_Date is earlier than Created_Date.")
        df.loc[invalid_deadlines, 'Deadline_Date'] = pd.NaT
print("\nDate Columns Status Summary:")
for col in date_cols:
    if col in df.columns:
        missing = df[col].isnull().sum()
        print(f" - {col}: {missing} missing values remaining.")

--- Step 8: Cleaning Date Columns ---
Fixing 143 records where Deadline_Date is earlier than Created_Date.

Date Columns Status Summary:
 - Date: 0 missing values remaining.
 - Created_Date: 0 missing values remaining.
 - Deadline_Date: 10153 missing values remaining.


---

## Step 9 — Clean Text Columns

Prepare the raw text fields so they are consistent and ready for **NLP Feature Engineering** (TF-IDF, embeddings) in the next notebook.

**Target columns:**
- `Task_Name`
- `Work_Description`
- `Task_Description`
- `Timesheet_Work_Logs`

### Cleaning Steps Applied

| Step | Action | Reason |
|---|---|---|
| 1 | `fillna("")` + cast to `str` | NLP pipelines require strings, not `NaN` or `float` |
| 2 | `.str.lower()` | Standardise case — `"Odoo"` and `"odoo"` should be treated as the same token |
| 3 | Replace `\n`, `\r`, `\t` with space | Remove formatting characters from exported ERP data |
| 4 | Replace multiple spaces with single space (`\s+` → `' '`) | Normalise inconsistent whitespace |
| 5 | `.str.strip()` | Remove leading/trailing whitespace |

> ⚠️ **TF-IDF is NOT applied here.** TF-IDF vectorisation belongs in the **Feature Engineering notebook**, where the vocabulary and weights will be fitted on the training set only (to prevent leakage from the test set into the vectoriser).

> 📌 **EDA Reference**: EDA Step 8 (Text Data Analysis) confirmed the presence of inconsistent casing, newline characters from ERP exports, and excessive whitespace in all four text columns.

In [15]:
import re
print("--- Step 9: Cleaning Text Columns ---")
text_cols = [
    'Task_Name', 
    'Work_Description', 
    'Task_Description', 
    'Timesheet_Work_Logs'
]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)
        df[col] = df[col].str.lower()
        df[col] = df[col].str.replace(r'[\n\r\t]', ' ', regex=True)
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True)
        df[col] = df[col].str.strip()
print("Text cleaning complete. Sample of cleaned 'Work_Description':")
if 'Work_Description' in df.columns:
    print(df['Work_Description'].head())

--- Step 9: Cleaning Text Columns ---
Text cleaning complete. Sample of cleaned 'Work_Description':
0                        db backup and restore in test
1                        db backup and restore in test
2                                           add addons
3                                                    /
4    get privillages to charith's new github accoun...
Name: Work_Description, dtype: str


---

## Step 10 — Validate Employee and Task Identifiers

Verify that the ID columns are internally consistent — that the same ID always maps to the same real-world entity across the dataset.

### Validations Performed

**Employee consistency** — for each `Employee_ID`, check:
- Does it always map to the same `Employee_Name`?
- Does it always map to the same `Employee_Department`?
- Does it always map to the same `Employee_Job_Position`?

**Task → Employee relationship** — for each `Task_ID`, check:
- How many unique `Employee_ID`s are associated with it?
- Multi-employee tasks are **expected** (collaborative tasks); single-employee tasks are also expected.

### Expected Outcomes

| Check | Expected Result | Action if Violated |
|---|---|---|
| Each `Employee_ID` → 1 name, 1 dept, 1 position | ✅ Consistent | Investigate and document |
| `Task_ID` → multiple employees | ✅ Expected for collaborative tasks | No action needed |
| `Task_ID` → 0 employees | ⚠️ Unexpected | Investigate |

> 📌 **EDA Reference**: EDA Step 10 (Correlation & Relationships) and EDA Step 7 (Employee–Task Relationship Analysis) established the expected many-to-many relationship between tasks and employees.

In [16]:
print("--- Step 10: Validating Identifiers ---")
emp_consistency = df_raw.groupby('Employee_ID').agg(
    Unique_Names=('Employee_Name', 'nunique'),
    Unique_Depts=('Employee_Department', 'nunique'),
    Unique_Jobs=('Employee_Job_Position', 'nunique')
)
inconsistent_emps = emp_consistency[
    (emp_consistency['Unique_Names'] > 1) | 
    (emp_consistency['Unique_Depts'] > 1) | 
    (emp_consistency['Unique_Jobs'] > 1)
]
if not inconsistent_emps.empty:
    print(f"⚠️ Warning: Found {len(inconsistent_emps)} employees with inconsistent profile mappings!")
    print(inconsistent_emps)
else:
    print("✅ Employee_ID mappings (Name, Department, Job Position) are completely consistent.")
task_collaborators = df.groupby('Task_ID')['Employee_ID'].nunique()
multi_employee_tasks = task_collaborators[task_collaborators > 1]
print(f"\n--- Task Assignment Verification ---")
print(f"Total unique tasks logged: {task_collaborators.count()}")
print(f"Tasks handled by a single employee: {sum(task_collaborators == 1)}")
print(f"Tasks handled by multiple employees (collaborations): {len(multi_employee_tasks)}")
if len(multi_employee_tasks) > 0:
    print(f"Maximum number of collaborators on a single task: {multi_employee_tasks.max()}")

--- Step 10: Validating Identifiers ---
✅ Employee_ID mappings (Name, Department, Job Position) are completely consistent.

--- Task Assignment Verification ---
Total unique tasks logged: 2442
Tasks handled by a single employee: 2000
Tasks handled by multiple employees (collaborations): 442
Maximum number of collaborators on a single task: 29


---

## Step 11 — Handle Data Leakage Risks

This is a critical step for the project. **Data leakage** occurs when information that would not be available at prediction time is used as a model input, causing artificially inflated performance during training that does not generalise to real-world use.

### The Core Question

> *"Would this information exist at the exact moment the manager wants to assign a new task?"*

```
New task arrives
      ↓
Manager wants a recommendation
      ↓
Model predicts the most suitable employee
      ↓
Task is assigned  ← this is where our model operates
```

At the point of prediction, **none of the following exist yet**:

| Column | Why It's a Leakage Risk |
|---|---|
| `Actual_Hours_Spent` | Post-task metric — unknown until the task is fully completed |
| `Task_Stage` | Reflects the current/completed status — unknown at assignment time |
| `Timesheet_Logs_Count` | Aggregated after timesheets are submitted — post-assignment |
| `Timesheet_Work_Logs` | Work descriptions written **after** the work is performed |
| `All_Collaborating_Employees` | Unknown until everyone has logged time on the task |

### Action Taken

Rather than silently dropping these columns now, they are **renamed** with a `FLAG_LEAKAGE_` prefix:

```
Actual_Hours_Spent  →  FLAG_LEAKAGE_Actual_Hours_Spent
Task_Stage          →  FLAG_LEAKAGE_Task_Stage
... etc.
```

This makes them **visible and documented** in the dataset. They will be explicitly excluded before the train/test split in the Feature Engineering notebook.

> 📌 **EDA Reference**: Leakage risks were originally identified and discussed in EDA Step 9 (Target Variable Analysis) and confirmed in EDA Step 11 (Data Integration Validation).

In [17]:
print("--- Step 11: Flagging Data Leakage Risks ---")

leakage_columns = [
    "Hours_Spent",
    "Work_Description",
    "Actual_Hours_Spent",
    "Task_Stage",
    "Timesheet_Logs_Count",
    "Timesheet_Work_Logs",
    "All_Collaborating_Employees",
]

rename_mapping = {
    col: f"FLAG_LEAKAGE_{col}"
    for col in leakage_columns
    if col in df.columns
}

df.rename(columns=rename_mapping, inplace=True)

print(
    "The following columns have been RED-FLAGGED for leakage "
    "and must never be model features:"
)

for old_name, new_name in rename_mapping.items():
    print(f"  {old_name} ---> {new_name}")

print(
    f"\nDataset Shape remains: "
    f"{df.shape[0]} rows, {df.shape[1]} columns"
)


--- Step 11: Flagging Data Leakage Risks ---
The following columns have been RED-FLAGGED for leakage and must never be model features:
  Hours_Spent ---> FLAG_LEAKAGE_Hours_Spent
  Work_Description ---> FLAG_LEAKAGE_Work_Description
  Actual_Hours_Spent ---> FLAG_LEAKAGE_Actual_Hours_Spent
  Task_Stage ---> FLAG_LEAKAGE_Task_Stage
  Timesheet_Logs_Count ---> FLAG_LEAKAGE_Timesheet_Logs_Count
  Timesheet_Work_Logs ---> FLAG_LEAKAGE_Timesheet_Work_Logs
  All_Collaborating_Employees ---> FLAG_LEAKAGE_All_Collaborating_Employees

Dataset Shape remains: 11630 rows, 19 columns


---

## Step 12 — Final Data Quality Validation

After all preprocessing steps, run a comprehensive final check and compare the dataset **before vs after** preprocessing.

### Validation Checklist

| Check | What We Verify |
|---|---|
| Shape | Row and column counts before vs after |
| Missing values | Should be zero (except `NaT` in `Deadline_Date`) |
| Duplicates | Should be zero completely duplicated rows |
| Data types | Dates as `datetime64`, numerics as `float64`, IDs as `str` |
| Unique employee count | Should match the original dataset |
| Unique task count | Should match or be slightly reduced |
| Numerical ranges | No negatives, `Hours_Spent` ≤ 24 |
| Categorical consistency | No unexpected `nan`/`None` strings |
| Text completeness | All text columns have no `NaN` (empty strings instead) |
| ID consistency | `Employee_ID` consistently maps to name/dept/position |

### Before vs After Summary

```
Before Preprocessing:  11,633 rows × 22 columns
          ↓
     Preprocessing Steps 1–11
          ↓
After Preprocessing:   XX rows × XX columns  ← filled in after running
```

In [18]:
print("==================================================")
print("     STEP 12: FINAL DATA QUALITY VALIDATION")
print("==================================================\n")
print("--- 1. Dataset Shape ---")
print(f"Before Preprocessing: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print("          ↓")
print("     Preprocessing")
print("          ↓")
print(f"After Preprocessing:  {df.shape[0]:,} rows × {df.shape[1]} columns\n")
print("--- 2. Missing-Value Summary ---")
missing_after = df.isnull().sum()
missing_filtered = missing_after[missing_after > 0]
if missing_filtered.empty:
    print("✅ No missing values remaining (excluding valid NaT in dates).")
else:
    print("Expected missing values (e.g., open-ended Deadline_Dates):")
    print(missing_filtered.to_string())
print("\n")
print("--- 3. Duplicate Summary ---")
print(f"Completely duplicated rows remaining: {df.duplicated().sum()} ✅\n")
print("--- 4. Data Types ---")
print(df.dtypes.value_counts().to_string())
print("\n")
print("--- 5. Unique Identifiers ---")
print(f"Unique Employee Count: {df['Employee_ID'].nunique()}")
print(f"Unique Task Count:     {df['Task_ID'].nunique()}\n")
print("--- 6. Numerical Ranges (Sample) ---")
num_cols = ['Hours_Spent', 'Estimated_Planned_Hours']
for col in num_cols:
    if col in df.columns:
        print(f"{col}: Min = {df[col].min()}, Max = {df[col].max()}, Mean = {df[col].mean():.2f}")
print("\n")
print("--- 7. Categorical Consistency ---")
cat_cols = ['Task_Priority', 'Employee_Department']
for col in cat_cols:
    if col in df.columns:
        print(f"{col} unique values: {df[col].nunique()} (Standardized to Title Case)")
print("\n")
print("--- 8. Text Completeness ---")
text_cols = ['Task_Name', 'Work_Description']
for col in text_cols:
    if col in df.columns:
        empty_strings = (df[col] == "").sum()
        print(f"{col}: {empty_strings} empty strings remaining (Ready for NLP)")
print("\n")
print("--- 9. ID Consistency ---")
print(f"Null Employee_IDs (Target): {df['Employee_ID'].isnull().sum()} ✅")
print(f"Null Task_IDs (Join Key):   {df['Task_ID'].isnull().sum()} ✅")
print("\n==================================================")
print("        PREPROCESSING COMPLETION SUMMARY")
print("==================================================")

     STEP 12: FINAL DATA QUALITY VALIDATION

--- 1. Dataset Shape ---
Before Preprocessing: 11,633 rows × 22 columns
          ↓
     Preprocessing
          ↓
After Preprocessing:  11,630 rows × 19 columns

--- 2. Missing-Value Summary ---
Expected missing values (e.g., open-ended Deadline_Dates):
Deadline_Date    10153


--- 3. Duplicate Summary ---
Completely duplicated rows remaining: 1 ✅

--- 4. Data Types ---
str               12
datetime64[us]     3
float64            3
int64              1


--- 5. Unique Identifiers ---
Unique Employee Count: 49
Unique Task Count:     2442

--- 6. Numerical Ranges (Sample) ---
Estimated_Planned_Hours: Min = 0.0, Max = 552.0, Mean = 10.16


--- 7. Categorical Consistency ---
Task_Priority unique values: 2 (Standardized to Title Case)
Employee_Department unique values: 8 (Standardized to Title Case)


--- 8. Text Completeness ---
Task_Name: 0 empty strings remaining (Ready for NLP)


--- 9. ID Consistency ---
Null Employee_IDs (Target): 0 ✅
Null

In [19]:
print("--- Preprocessed Dataset (First 5 Rows) ---")
display(df.head())

--- Preprocessed Dataset (First 5 Rows) ---


,Date,Task_ID,Task_Name,FLAG_LEAKAGE_Work_Description,FLAG_LEAKAGE_Hours_Spent,Project_Name,Employee_ID,Employee_Department,Employee_Job_Position,Task_Description,FLAG_LEAKAGE_Timesheet_Work_Logs,Task_Priority,Estimated_Planned_Hours,FLAG_LEAKAGE_Actual_Hours_Spent,FLAG_LEAKAGE_Timesheet_Logs_Count,FLAG_LEAKAGE_Task_Stage,Created_Date,Deadline_Date,FLAG_LEAKAGE_All_Collaborating_Employees
0,2026-09-14,TSK-1640,odoo sh maintain,db backup and restore in test,0.25,Hovael Project,EMP-11,Research And Development (R&D),Team Lead - Research And Development (R&D),odoo sh maintain. work details: add addons t t...,add addons t the server | get backup from live...,Low,0.0,14.25,27,Project Preparation,2026-02-05,NaT,W M I L Wijesinghe
1,2026-09-14,TSK-1881,odoo sh maintain,db backup and restore in test,0.25,Ceylon Eco Spices,EMP-11,Research And Development (R&D),Team Lead - Research And Development (R&D),odoo sh maintain. work details: add addon and ...,add addon and test | get odoo sh live ackup an...,Low,0.0,6.50,13,Developments,2026-03-19,NaT,W M I L Wijesinghe
2,2026-09-14,TSK-182,odoo.sh maintaing,add addons,0.25,Mihiri Bakemart (Pvt)Ltd - Development,EMP-11,Research And Development (R&D),Team Lead - Research And Development (R&D),odoo.sh maintaing. work details: add all addon...,add all addon and build and test on the odoo s...,Low,0.0,40.75,29,Ongoing,2025-06-09,NaT,W M I L Wijesinghe
3,2026-09-14,TSK-2884,development meeting,/,0.00,Cygnus One,EMP-55,Colombo Branch,Project Manager,development meeting. work details: intern deve...,intern development hoveal project meeting | me...,Low,0.0,9.42,13,Miscellaneous,2026-07-06,NaT,"W M I L Wijesinghe, K R V Dias, A R M S Madusa..."
4,2026-09-14,TSK-405,other tasks (mention on description),get privillages to charith's new github accoun...,0.50,Cygnus One,EMP-11,Research And Development (R&D),Team Lead - Research And Development (R&D),other tasks (mention on description). work det...,self ssl setup on vps | preparing report list ...,Low,0.0,162.18,69,Miscellaneous,2025-07-14,NaT,"H.M.C.S Thilakarathna, Sadaruwan Bandara, L H ..."


---

## Step 13 — Save the Preprocessed Dataset

Save the clean, preprocessed dataset to `data/processed/` so it can serve as the **input for the Feature Engineering notebook** (`04_Feature_Engineering.ipynb`).

**Output file:** `data/processed/preprocessed_employee_task_data.csv`

> ⚠️ The `data/processed/` directory is excluded from version control via `.gitignore` because it contains derived data from the raw organisational dataset.

In [20]:
import os
print("--- Step 13: Saving the Preprocessed Dataset ---")
output_dir = '../data/processed'
output_file = 'preprocessed_employee_task_data.csv'
output_path = os.path.join(output_dir, output_file)
df.to_csv(output_path, index=False)
print(f"✅ Preprocessed dataset successfully saved to: {output_path}")
print("Your data is officially ready for the Feature Engineering notebook!")

--- Step 13: Saving the Preprocessed Dataset ---
✅ Preprocessed dataset successfully saved to: ../data/processed/preprocessed_employee_task_data.csv
Your data is officially ready for the Feature Engineering notebook!


---

## ✅ What Has Been Done in This Notebook

| Step | Task | Status |
|---|---|---|
| 1 | Loaded combined dataset & created working copy | ✅ Done |
| 2 | Removed redundant columns (`Timesheet_ID`, `Employee_Name`, `Original_Task_Description`) | ✅ Done |
| 2 | Flagged leakage-risk columns (`Actual_Hours_Spent`, `Task_Stage`, etc.) | ✅ Done |
| 3 | Handled missing values per column type (text → `""`, categorical → `"Unknown"`, numeric → `0.0`) | ✅ Done |
| 4 | Fixed data types (dates → `datetime64`, numerics → `float64`, IDs → `str`) | ✅ Done |
| 5 | Checked and removed fully duplicated rows (no `Task_ID` deduplication) | ✅ Done |
| 6 | Standardised categorical variables (strip, title-case, spelling maps) | ✅ Done |
| 7 | Cleaned numerical variables (negatives → abs, `Hours_Spent` capped at 24) | ✅ Done |
| 8 | Cleaned date columns (invalid dates → `NaT`, chronological errors fixed) | ✅ Done |
| 9 | Cleaned text columns (lowercase, strip, whitespace normalisation) | ✅ Done |
| 10 | Validated employee and task identifier consistency | ✅ Done |
| 11 | Flagged leakage-risk columns with `FLAG_LEAKAGE_` prefix | ✅ Done |
| 12 | Final data quality validation (before vs after comparison) | ✅ Done |
| 13 | Saved preprocessed dataset to `data/processed/preprocessed_employee_task_data.csv` | ✅ Done |
| 14 | Dropped leakage-flagged columns & split into Train (60%) / Val (20%) / Test (20%) with stratification | ✅ Done |
| 14 | Saved `train_data.csv`, `val_data.csv`, `test_data.csv` to `data/processed/` | ✅ Done |

---

## 🔲 What Comes Next — Feature Engineering Notebook

The next notebook (`04_Feature_Engineering.ipynb`) will use the preprocessed dataset to build the feature set for the ML models:

| Task | Detail |
|---|---|
| Drop leakage columns | Remove all `FLAG_LEAKAGE_*` columns before splitting |
| Train / test split | Split on time or randomly with stratification |
| TF-IDF | Apply to `Task_Description`, `Work_Description`, `Task_Name` (fit on train only) |
| Employee aggregate features | Total tasks, total hours, tasks-per-type per employee |
| Employee–task relationship features | Has this employee done this task type before? How many times? |
| Date-derived features | Task age, `has_deadline` flag, month, day-of-week |
| Collaborator count | Derive from `FLAG_LEAKAGE_All_Collaborating_Employees` (task-level only, not as model input) |
| Encoding | Label-encode or one-hot encode categorical features |
| Normalisation | Scale numerical features |